# delbina-sum: GGUF export + re-eval (Kaggle, GPU T4 x2)
Settings (right panel): Accelerator **GPU T4 x2**, Internet **on**, Add-ons → Secrets → `HF_TOKEN` attached.
Run All. Cell 2 (build) ≈ 2.5–3.5 h, cells 3–5 ≈ 1 h. Keep the tab open in a live session.
If the session restarts: run cell 1, then continue from the cell that was running (the build resumes, nothing is lost on disk).
Cell 5 uploads only quants that pass every bar, to the private repo `delbina/delbina-sum-GGUF`.
Holdout: read only the aggregate lines this prints.

In [ ]:
# 1. setup (~2 min). Run this first again after any restart.
import os
os.chdir("/kaggle/working")
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["LD_LIBRARY_PATH"] = "/usr/local/nvidia/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")  # Kaggle's libcuda
REV = "d33bbb227b774f85dde6d4d9b13037e62ed736f7"   # delbina/delbina-sum@v5-1.5b, holdout aggregates
QUANTS = ["Q8_0", "Q4_K_M"]
print("token loaded:", bool(os.environ["HF_TOKEN"]))
!nvidia-smi --query-gpu=name,memory.total --format=csv
!nproc; free -g | head -2
!test -d delbina-sum || git clone -q https://github.com/delbina/delbina-sum
!test -d llama.cpp || git clone -q --depth 1 https://github.com/ggml-org/llama.cpp
!pip -q install ./llama.cpp/gguf-py delbina==0.4.0   # gguf: converter; delbina: datagen imports it (pinned in requirements.txt)
!git -C delbina-sum log --oneline -1
!git -C llama.cpp log --oneline -1

In [ ]:
# 2. build llama.cpp with CUDA for the T4 (~2.5-3.5 h). Safe to re-run: it resumes.
# - libcuda lives in /usr/local/nvidia/lib64 on Kaggle, where CMake does not look: find it and pass it in
# - -j 3, not plain -j: plain -j starts unlimited nvcc jobs and runs out of RAM
import os, subprocess, time
found = subprocess.run("find /usr -name 'libcuda.so*' 2>/dev/null", shell=True, capture_output=True, text=True).stdout.split()
cands = sorted(found, key=lambda p: ("/usr/local/nvidia/" not in p, "stubs" not in p, len(p)))
print("libcuda:", cands[:3])
assert cands, "no libcuda found"
cfg = subprocess.run(["cmake", "-S", "llama.cpp", "-B", "llama.cpp/build", "-DGGML_CUDA=ON",
                      f"-DCUDA_cuda_driver_LIBRARY={cands[0]}", "-DCMAKE_CUDA_ARCHITECTURES=75",
                      "-DLLAMA_CURL=OFF", "-DLLAMA_BUILD_TESTS=OFF"], capture_output=True, text=True)
open("configure.log", "w").write(cfg.stdout + cfg.stderr)
print("\n".join(l for l in (cfg.stdout + cfg.stderr).splitlines() if "CUDA" in l or "rror" in l))
assert cfg.returncode == 0, "configure failed, see configure.log"
t = time.time()
!cmake --build llama.cpp/build -j 3 --target llama-quantize llama-server > build.log 2>&1; tail -3 build.log
print(f"build: {(time.time() - t) / 60:.0f} min")
for b in ("llama-quantize", "llama-server"):
    assert os.path.exists(f"llama.cpp/build/bin/{b}"), f"build failed: {b} missing, see build.log"
print("BUILD OK")

In [ ]:
# 3. download the pinned merged model, convert to GGUF, quantize (~5-10 min)
from huggingface_hub import snapshot_download
snapshot_download("delbina/delbina-sum", revision=REV, local_dir="merged",
                  allow_patterns=["*.json", "*.safetensors", "*.txt", "*.jinja"],
                  ignore_patterns=["adapter/*", "results.json", "holdout_results.json"])
!python llama.cpp/convert_hf_to_gguf.py merged --outtype f16 --outfile delbina-sum-1.5b-f16.gguf > convert.log 2>&1; tail -2 convert.log
assert os.path.exists("delbina-sum-1.5b-f16.gguf"), "convert failed, see convert.log"
for q in QUANTS:
    !llama.cpp/build/bin/llama-quantize delbina-sum-1.5b-f16.gguf delbina-sum-1.5b-{q}.gguf {q} > quant_{q}.log 2>&1; tail -1 quant_{q}.log
    assert os.path.exists(f"delbina-sum-1.5b-{q}.gguf"), f"quantize {q} failed, see quant_{q}.log"
!ls -lh *.gguf

In [ ]:
# 4. eval each quant on test + holdout with the same metrics and bars (~20-30 min per quant)
import os, sys, json, time, subprocess, urllib.request
from concurrent.futures import ThreadPoolExecutor
sys.path[:0] = ["delbina-sum/datagen", "delbina-sum/train"]
import make_dataset as md0, make_dataset_v2 as md2, sft
from common import clean_summary
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("merged")
N, MAX_NEW, PAR = 300, 384, 8
SETS = {"test-v2": [md2.make_sample("test", i) for i in range(N)],
        "test-v0": [md0.make_sample("test", i) for i in range(N)],
        "holdout-v2": [md2.make_sample("holdout", i) for i in range(N)],
        "holdout-v0": [md0.make_sample("holdout", i) for i in range(N)]}
UNSEEN = md2.TEST_KINDS | md0.TEST_KINDS
# fp16 numbers from the Kaggle run: (fact_recall, unseen_recall, invented_rate)
REF = {"test-v2": (98.0, 97.2, 0.6), "test-v0": (92.7, 86.3, 0.5),
       "holdout-v2": (97.8, 97.6, 0.6), "holdout-v0": (90.6, 87.3, 0.5)}
BASE_RECALL = 22.5  # plain Qwen2.5-1.5B on test-v2


def serve(gguf):
    env = dict(os.environ, CUDA_VISIBLE_DEVICES="0")  # one T4 is plenty for 1.5B; no layer split
    p = subprocess.Popen(["llama.cpp/build/bin/llama-server", "-m", gguf, "-ngl", "99",
                          "-c", str(PAR * 8192), "-np", str(PAR), "--port", "8080"],
                         stdout=subprocess.DEVNULL, stderr=open("server.log", "w"), env=env)
    for _ in range(180):
        try:
            if urllib.request.urlopen("http://127.0.0.1:8080/health", timeout=2).status == 200:
                return p
        except Exception:
            time.sleep(1)
    p.kill()
    print(open("server.log").read()[-3000:])
    raise RuntimeError("llama-server did not start (log above)")


def gen(prompt):
    body = json.dumps({"prompt": prompt, "n_predict": MAX_NEW, "temperature": 0, "top_k": 1,
                       "repeat_penalty": 1.0, "cache_prompt": False,
                       "stop": ["<|im_end|>", "<|endoftext|>"]}).encode()
    req = urllib.request.Request("http://127.0.0.1:8080/completion", body, {"Content-Type": "application/json"})
    r = json.load(urllib.request.urlopen(req, timeout=900))
    return {"summary": clean_summary(r["content"]), "truncated": r["tokens_predicted"] >= MAX_NEW}


def bars(r):
    t2, t0, h2, h0 = r["test-v2"], r["test-v0"], r["holdout-v2"], r["holdout-v0"]
    return {"test: recall gain >= +15": t2["fact_recall"] - BASE_RECALL >= 15,
            "test: invented <= 2.0%": t2["invented_rate"] <= 2.0,
            "test: unseen recall >= 70": (t2["unseen_recall"] or 0) >= 70,
            "test: distractor hits <= 5%": (t2["distractor_hit_rate"] or 0) <= 5,
            "test v0: unseen recall >= 60": (t0["unseen_recall"] or 0) >= 60,
            "test v0: invented <= 2.0%": t0["invented_rate"] <= 2.0,
            "holdout: invented <= 2.0%": h2["invented_rate"] <= 2.0,
            "holdout: unseen recall >= 70": (h2["unseen_recall"] or 0) >= 70,
            "holdout: distractor hits <= 5%": (h2["distractor_hit_rate"] or 0) <= 5,
            "holdout v0: unseen recall >= 60": (h0["unseen_recall"] or 0) >= 60,
            "holdout v0: invented <= 2.0%": h0["invented_rate"] <= 2.0}


ALL = {}
for q in QUANTS:
    srv = serve(f"delbina-sum-1.5b-{q}.gguf")
    try:
        res = {}
        for name, rows in SETS.items():
            t = time.time()
            with ThreadPoolExecutor(PAR) as ex:
                outs = list(ex.map(gen, [sft._prompt_text(tok, r["lines"]) for r in rows]))
            res[name] = sft.evaluate(rows, outs, UNSEEN)
            m, ref = res[name], REF[name]
            drift = (m["fact_recall"] < ref[0] - 2) or (m["invented_rate"] > ref[2] + 0.5)
            print(f"{q} {name}: {time.time() - t:.0f}s  recall {m['fact_recall']} (fp16 {ref[0]})  "
                  f"unseen {m['unseen_recall']} (fp16 {ref[1]})  invented {m['invented_rate']} (fp16 {ref[2]})  "
                  f"distractor {m['distractor_hit_rate']}  truncated {m['truncated']}" + ("  <-- DRIFT" if drift else ""),
                  flush=True)
    finally:
        srv.kill()
        srv.wait()
    checks = bars(res)
    ALL[q] = {"results": res, "checks": checks, "pass": all(checks.values())}
    print(f"\n{q} verdict:")
    for k, v in checks.items():
        print(f"  {'PASS' if v else 'FAIL'}  {k}")
    print(f"{q}: {'GGUF PASS' if ALL[q]['pass'] else 'GGUF FAIL'}\n", flush=True)

json.dump({"source_revision": REV, **ALL}, open("gguf_results.json", "w"), indent=1)
print("saved gguf_results.json")


In [ ]:
# 5. upload passing quants to a separate private repo (skips any quant that failed a bar)
from huggingface_hub import HfApi
api = HfApi()
REPO_GGUF = "delbina/delbina-sum-GGUF"
api.create_repo(REPO_GGUF, private=True, exist_ok=True)
for q, v in ALL.items():
    if v["pass"]:
        api.upload_file(path_or_fileobj=f"delbina-sum-1.5b-{q}.gguf", path_in_repo=f"delbina-sum-1.5b-{q}.gguf",
                        repo_id=REPO_GGUF, commit_message=f"{q} from delbina-sum@{REV[:7]}, all bars pass")
        print("uploaded", q)
    else:
        print("skipped (failed bars):", q)
api.upload_file(path_or_fileobj="gguf_results.json", path_in_repo="gguf_results.json",
                repo_id=REPO_GGUF, commit_message="eval results for the quants")
print("https://huggingface.co/" + REPO_GGUF)